# XGBoost ranker

Read the fixed trip folds and shared features. Use nested, trip-based inner holdouts for Optuna tuning; outer validation trips never enter tuning or fitting.

Tuning uses at most 10,000 complete training groups and 3,000 complete validation groups per study to bound runtime. Each outer model then fits **all eligible groups** from its four training folds. Five Optuna trials per study is a small starting search, not an exhaustive optimization.

Keep singletons and candidate misses for held-out predictions and scoring. Exclude them only from ranker training. This notebook does not run the other base models, stacking or student-test scoring.

In [1]:
%run 06_model_helpers.ipynb
import optuna
import xgboost as xgb
from sklearn.model_selection import GroupShuffleSplit
from time import perf_counter
from datetime import datetime
from zoneinfo import ZoneInfo

n_trials = 5
tuning_train_groups = 10000
tuning_valid_groups = 3000
n_jobs = 4
optuna.logging.set_verbosity(optuna.logging.WARNING)
started = datetime.now(ZoneInfo("Asia/Kolkata"))
run_id = started.strftime("%Y%m%d_%H%M%S")
output = Path("results/xgb_ranker") / run_id
output.mkdir(parents=True, exist_ok=True)
data, points = load_model_data()
print("Labelled candidate rows:", len(data), "| Valid points:", len(points))

/Users/admin/.pyenv/versions/3.11.9/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Labelled candidate rows: 11502356 | Valid points: 206332


## Fit and tune

Ranker groups are built after every split. They contain at least two candidates and exactly one positive. Group sizes follow the sorted row order passed to XGBoost.

Each Optuna objective is Top-1 on an inner trip holdout, including its candidate misses. Sampling selects whole GPS groups. Fixed seed and input order make those samples reproducible.

In [2]:
def fit_ranker(frame, params):
    sizes = frame.groupby("group_id")["label"].agg(["size", "sum"])
    eligible = sizes.index[(sizes["size"] >= 2) & (sizes["sum"] == 1)]
    training = frame[frame["group_id"].isin(eligible)]
    assert training["group_id"].is_monotonic_increasing
    groups = training.groupby("group_id", sort=False).size().to_numpy()
    model = xgb.XGBRanker(objective="rank:ndcg", eval_metric="ndcg@1", tree_method="hist",
                          random_state=seed, n_jobs=n_jobs, **params)
    model.fit(training[feature_columns], training["label"], group=groups, verbose=False)
    return model


def tune_ranker(frame, point_frame, split_seed, filename):
    group_rows = frame[["group_id", "trip_id"]].drop_duplicates("group_id")
    train_idx, valid_idx = next(GroupShuffleSplit(1, test_size=0.2, random_state=split_seed).split(group_rows, groups=group_rows["trip_id"]))
    train_ids = group_rows.iloc[train_idx]["group_id"].sample(n=min(tuning_train_groups, len(train_idx)), random_state=split_seed)
    valid_ids = group_rows.iloc[valid_idx]["group_id"].sample(n=min(tuning_valid_groups, len(valid_idx)), random_state=split_seed)
    inner_train = frame[frame["group_id"].isin(train_ids)]
    inner_valid = frame[frame["group_id"].isin(valid_ids)]
    inner_points = point_frame[point_frame["group_id"].isin(valid_ids)]
    assert set(inner_train["trip_id"]).isdisjoint(inner_valid["trip_id"])

    def objective(trial):
        params = {"n_estimators": trial.suggest_categorical("n_estimators", [50, 100]),
                  "max_depth": trial.suggest_int("max_depth", 3, 6),
                  "learning_rate": trial.suggest_float("learning_rate", 0.05, 0.3, log=True),
                  "min_child_weight": trial.suggest_float("min_child_weight", 1, 10),
                  "subsample": trial.suggest_float("subsample", 0.7, 1),
                  "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1),
                  "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 10, log=True)}
        model = fit_ranker(inner_train, params)
        probability = point_softmax(model.predict(inner_valid[feature_columns]), inner_valid["group_id"])
        return score_points(inner_valid, probability, inner_points)["top1"]

    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=split_seed))
    study.optimize(objective, n_trials=n_trials)
    study.trials_dataframe().to_csv(output / filename, index=False)
    return study.best_params

## Five outer folds and OOF predictions

Every valid held-out candidate gets a score, including groups with one candidate or no positive. Stable per-point softmax produces the common OOF probability column. Fit time excludes tuning; prediction time includes softmax.

In [3]:
raw_oof = np.full(len(data), np.nan, dtype=np.float32)
probability_oof = np.full(len(data), np.nan, dtype=np.float32)
records = []
fold_params = {}
for fold in range(5):
    training = data[data["fold"] != fold]
    validation = data[data["fold"] == fold]
    train_points = points[points["fold"] != fold]
    valid_points = points[points["fold"] == fold]
    assert set(training["trip_id"]).isdisjoint(validation["trip_id"])
    start = perf_counter()
    params = tune_ranker(training, train_points, seed + fold, f"tuning_fold_{fold}.csv")
    tuning_s = perf_counter() - start
    fold_params[fold] = params
    start = perf_counter()
    model = fit_ranker(training, params)
    fit_s = perf_counter() - start
    start = perf_counter()
    raw = model.predict(validation[feature_columns])
    probability = point_softmax(raw, validation["group_id"])
    predict_s = perf_counter() - start
    raw_oof[validation.index] = raw
    probability_oof[validation.index] = probability
    metrics = score_points(validation, probability, valid_points)
    records.append({"fold": fold, **metrics, "tuning_s": tuning_s, "fit_s": fit_s, "predict_s": predict_s})
    pd.DataFrame(records).to_csv(output / "progress.csv", index=False)
    print("Fold", fold, "| Top-1:", metrics["top1"], "| Fit seconds:", round(fit_s, 1), flush=True)

assert np.isfinite(raw_oof).all() and np.isfinite(probability_oof).all()
oof = data[["group_id", "segment_id", "fold"]].copy()
oof["score"] = probability_oof
oof["raw_score"] = raw_oof
oof.to_csv(output / "oof_xgb_ranker.csv", index=False)
cv = pd.DataFrame(records)
cv.to_csv(output / "cv_metrics.csv", index=False)
pooled = score_points(data, probability_oof, points)
(output / "pooled_metrics.json").write_text(json.dumps(pooled, indent=2) + "\n")
metric_names = ["top1", "macro_f1", "precision", "recall", "log_loss", "candidate_coverage"]
display(cv[metric_names].agg(["mean", "std"]).T)
print("Pooled OOF:", pooled)

Fold 0 | Top-1: 0.512175143346489 | Fit seconds: 88.5


Fold 1 | Top-1: 0.8515855324656648 | Fit seconds: 95.3


Fold 2 | Top-1: 0.9224429163571891 | Fit seconds: 109.6


Fold 3 | Top-1: 0.8306525624305656 | Fit seconds: 124.0


Fold 4 | Top-1: 0.8989096913625215 | Fit seconds: 110.1


,mean,std
top1,0.803153,0.166710
macro_f1,0.900491,0.083267
precision,0.803153,0.166710
recall,0.805256,0.163025
log_loss,0.917794,0.972216
candidate_coverage,0.996215,0.007536


Pooled OOF: {'top1': 0.804892115619487, 'macro_f1': 0.9014275805807033, 'precision': 0.804892115619487, 'recall': 0.8078835227272727, 'log_loss': 0.9021553993225098, 'candidate_coverage': 0.9962972297074617, 'points': 206332, 'candidate_misses': 764, 'confusion_matrix': [[11256531, 40257], [39493, 166075]]}


## Final training fit and experiment row

Tune once more on an inner trip split of all training trips, then refit on all eligible training groups. Time that final fit separately. Student-test metrics stay empty until fixed-setting test candidates and features exist.

The nested model tuning keeps outer validation trips out of its inner fits. Earlier candidate and label thresholds were selected on all training trips, so CV is conditional on those previously fixed preprocessing choices. Per-point softmax is uncalibrated; no temperature is fitted on reported OOF labels.

In [4]:
start = perf_counter()
final_params = tune_ranker(data, points, seed + 100, "tuning_final.csv")
final_tuning_s = perf_counter() - start
start = perf_counter()
final_model = fit_ranker(data, final_params)
final_fit_s = perf_counter() - start
final_model.save_model(output / "model.json")
(output / "params.json").write_text(json.dumps({"fold_params": fold_params, "final_params": final_params}, indent=2) + "\n")
row = {"run_id": run_id, "timestamp": started.isoformat(), "model": "xgb_ranker",
       "params": json.dumps({"fold_params": fold_params, "final_params": final_params}), "seed": seed,
       "fold_scheme": contract["fold_scheme"] + "; nested inner trip holdout tuning",
       "radius_m": contract["candidate_settings"]["radius_m"], "K": contract["candidate_settings"]["K"],
       "fit_s_mean": cv["fit_s"].mean(), "predict_s_mean": cv["predict_s"].mean(),
       "tuning_s_total": cv["tuning_s"].sum() + final_tuning_s, "final_fit_s": final_fit_s,
       "test_status": "not_run: student-test labels/candidates/features not built", "output_dir": str(output)}
for name in metric_names:
    row[f"cv_{name}_mean"] = cv[name].mean()
    row[f"cv_{name}_std"] = cv[name].std()
    row[f"pooled_{name}"] = pooled[name]
    row[f"test_{name}"] = None
row["pooled_confusion_matrix"] = json.dumps(pooled["confusion_matrix"])
row["final_predict_s"] = None
experiments = Path("results/experiments.csv")
new = pd.DataFrame([row])
if experiments.exists():
    new = pd.concat([pd.read_csv(experiments), new], ignore_index=True)
new.to_csv(experiments, index=False)
print("Saved model, OOF predictions, fold metrics and experiment row to", output)

Saved model, OOF predictions, fold metrics and experiment row to results/xgb_ranker/20261118_121050
